#**DEPENDENCY INSTALLATION**

In [18]:
!pip install transformers datasets sentence-transformers scikit-learn accelerate evaluate rouge_score

# **SEMANTIC_COMMENT_CLUSTERING**

In [43]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# 1. Initialize the Embedding Model
# We use 'all-distilroberta-v1' as a fast, high-performance alternative
embedder = SentenceTransformer('all-distilroberta-v1')

def cluster_comments(comments):
    """
    Clusters comments based on semantic similarity.
    Uses a dynamic cluster count to avoid 'more clusters than samples' errors.
    """
    # Safety checks for empty or single-comment threads
    if not comments:
        return {}
    num_samples = len(comments)
    if num_samples == 1:
        return {0: comments}

    # A. Encode comments into vectors (embeddings)
    embeddings = embedder.encode(comments)

    # B. Perform Agglomerative Clustering
    # Logic: Aim for 8 clusters, but never more than the total number of comments.
    # We set distance_threshold=None to allow n_clusters to dictate the groups.
    n_to_use = min(8, num_samples)

    clustering_model = AgglomerativeClustering(
        n_clusters=n_to_use,
        distance_threshold=None,
        metric='cosine',
        linkage='average'
    )

    clustering_model.fit(embeddings)
    cluster_assignment = clustering_model.labels_

    # C. Group comments by their cluster ID
    clustered_comments = {}
    for idx, cluster_id in enumerate(cluster_assignment):
        cluster_id = int(cluster_id)
        if cluster_id not in clustered_comments:
            clustered_comments[cluster_id] = []
        clustered_comments[cluster_id].append(comments[idx])

    return clustered_comments

# ---------------------------------------------------------
# 2. LOAD REAL DATA FROM train.json
# ---------------------------------------------------------

# Load the JSON file
with open('/content/drive/MyDrive/data/downloadable_data/raw/train.json', 'r') as f:
    data = json.load(f)

# Select a thread (Change index to test different scenarios)
selected_thread = data['threads'][0]

print(f"Processing Thread ID: {selected_thread['submission_id']}")
print(f"Subreddit: {selected_thread['subreddit']}")
print(f"Original Caption: {selected_thread['caption']}\n")

# Extract comment bodies
real_comments = [c['body'] for c in selected_thread['comments']]

# Clean comments
real_comments = [c for c in real_comments if c not in ["[deleted]", "[removed]"] and c.strip() != ""]

print(f"Found {len(real_comments)} valid comments. Clustering now...\n")

# ---------------------------------------------------------
# 3. Run the Clustering
# ---------------------------------------------------------
# We skip the first comment usually if it's the AutoMod or OP clarification
op_clarification_text = real_comments[0]
community_comments = real_comments[1:]

clusters = cluster_comments(community_comments)

# ---------------------------------------------------------
# 4. Display Results
# ---------------------------------------------------------
print("-" * 30)
print(f"GENERATED {len(clusters)} CLUSTERS")
print("-" * 30)
for cluster_id, comment_list in clusters.items():
    print(f"CLUSTER {cluster_id} ({len(comment_list)} comments):")
    for comment in comment_list:
        preview = comment[:100] + "..." if len(comment) > 100 else comment
        print(f"  - {preview}")
    print("-" * 30)

Processing Thread ID: coiccq
Subreddit: designmyroom
Original Caption: would you put the sectional on the other side??

Found 8 valid comments. Clustering now...

------------------------------
GENERATED 7 CLUSTERS
------------------------------
CLUSTER 3 (1 comments):
  - No. In my humble opinion, the chaise part of the section always go along the wall/window side of a r...
------------------------------
CLUSTER 6 (1 comments):
  - Absolutely do not put the tv in front of your beautiful windows. This sofa won't work in this room n...
------------------------------
CLUSTER 5 (1 comments):
  - Honestly, I would sell the couch and buy a smaller one; it's too big for the room.
------------------------------
CLUSTER 4 (1 comments):
  - If you put it on the other side and are able to keep the chaise on that side it seems like you’ll co...
------------------------------
CLUSTER 1 (1 comments):
  - I’d mount the tv on the wall, not block the windows w it
------------------------------
CLUSTER

# **SUPERVISED_FINE_TUNING_ON_MREDDITSUM**

In [44]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, LongT5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate
import numpy as np

# 1. Load Data from Text Files
def load_data_from_text(src_file, tgt_file):
    with open(src_file, 'r', encoding='utf-8') as f:
        sources = [line.strip() for line in f.readlines()]
    with open(tgt_file, 'r', encoding='utf-8') as f:
        targets = [line.strip() for line in f.readlines()]
    return Dataset.from_dict({"text": sources, "summary": targets})

# 2. LOAD ALL THREE DATASETS (Train: 2729, Val: 152, Test: 152)
# Ensure these paths correctly point to your Drive files
base_path = '/content/drive/MyDrive/data/downloadable_data/preprocessed/'

train_dataset = load_data_from_text(f'{base_path}train_processed_src.txt', f'{base_path}train_processed_tgt.txt')
val_dataset = load_data_from_text(f'{base_path}val_processed_src.txt', f'{base_path}val_processed_tgt.txt')
test_dataset = load_data_from_text(f'{base_path}test_processed_src.txt', f'{base_path}test_processed_tgt.txt')

# Combine into a proper DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

# 3. Tokenizer & Model Setup
model_checkpoint = "google/long-t5-tglobal-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = LongT5ForConditionalGeneration.from_pretrained(model_checkpoint)

# 4. Preprocessing
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 5. Metrics
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 6. Updated Training Arguments (Matching paper's 50 epochs and 3e-5 learning rate)
args = Seq2SeqTrainingArguments(
    output_dir="./t5-mredditsum-final-splits",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # Paper's recommended rate [cite: 719]
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=20,         # Paper trained for 50 epochs
    predict_with_generate=True,
    bf16=True,
    load_best_model_at_end=True, # Critical: Uses validation set to pick best model
    metric_for_best_model="rouge1",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Training monitors validation performance
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

# 7. Train
trainer.train()

# 8. FINAL EVALUATION ON THE TEST SET
print("\n--- Final Evaluation on Unseen Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test ROUGE Scores: {test_results}")

# 9. Save
trainer.save_model("./final_mredditsum_long-t5-tglobal-base_without_img_caption_20_epoch")

Map:   0%|          | 0/2729 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

/tmp/ipython-input-3342616410.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.441300,1.764621,20.971400,11.368500,18.755800,18.746500
2,2.121900,1.718581,20.918400,11.794600,18.823300,18.803900
3,1.961400,1.689584,21.267800,12.022500,19.000300,18.976700
4,1.915600,1.673645,21.217400,11.794800,18.920200,18.879600
5,1.889300,1.670702,20.754000,11.597500,18.632600,18.601900
6,1.821800,1.666309,21.107900,11.801100,18.830000,18.803900
7,1.806600,1.660427,21.425300,12.212300,19.123300,19.080000
8,1.777700,1.655053,21.002100,12.015600,18.937000,18.893300
9,1.737900,1.647442,21.090200,12.051900,18.946500,18.956700
10,1.738200,1.653536,21.181400,11.832500,18.857500,18.837600



--- Final Evaluation on Unseen Test Set ---


Test ROUGE Scores: {'eval_loss': 1.7147222757339478, 'eval_rouge1': 20.5818, 'eval_rouge2': 10.99, 'eval_rougeL': 18.3374, 'eval_rougeLsum': 18.3603, 'eval_runtime': 26.3698, 'eval_samples_per_second': 5.764, 'eval_steps_per_second': 1.441, 'epoch': 20.0}


# **SINGLE_PASS_BASELINE_INFERENCE**

In [45]:
from transformers import pipeline

# Load your fine-tuned model
summarizer = pipeline("summarization", model="/content/final_mredditsum_long-t5-tglobal-base_without_img_caption_20_epoch", tokenizer=tokenizer, device=0)

# Example input (You can paste a raw line from your src.txt here)
input_text = "Original Post: What color should I paint my walls? Image: A living room with beige furniture. OP: I feel like it's too boring. User 1: Try sage green! User 2: I agree, green would look great."

# T5 prompt prefix
input_text = "summarize: " + input_text

summary = summarizer(input_text, max_length=128, min_length=30, do_sample=False)
print("Generated Summary:", summary[0]['summary_text'])

Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Summary: The OP asked what color they should paint the walls in their living room with beige furniture. One commenter suggested sage green for the walls. Another commenter suggested sage green for the walls.


# **CMS_PIPELINE_EXECUTION_(SUMMARIZATION_&_SYNTHESIS)**

In [46]:
def run_cms_pipeline(thread_json_object, model, tokenizer):
    """
    Uses the raw JSON object to ensure we cluster real COMMENTS, not words.
    """
    # 1. Extract real comments from JSON
    real_comments = [c['body'] for c in thread_json_object['comments']]
    real_comments = [c for c in real_comments if c not in ["[deleted]", "[removed]"] and c.strip() != ""]

    if len(real_comments) < 2:
        return "Not enough comments to summarize."

    # First comment is usually OP clarification
    op_clarification = real_comments[0]
    community_comments = real_comments[1:]
    op_caption = thread_json_object.get('caption', 'No image available')

    # 2. Stage 1: Clustering (Using your new dynamic function)
    clusters = cluster_comments(community_comments)

    # 3. Stage 2: Summarize Clusters
    op_sum, cluster_sums = stage_2_summarize_clusters_fixed(
        clusters, op_caption, op_clarification, model, tokenizer
    )

    # 4. Stage 3: Synthesis
    final_summary = stage_3_synthesis(op_sum, cluster_sums, model, tokenizer)

    return final_summary

In [51]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
cms_predictions = []
ground_truth_references = []

# Assuming you have loaded your JSON: data = json.load(f)
# We need to find where the 'test' threads start.
# Usually, in MREDDITSUM, the last 152 threads are the test set.
test_threads = data['threads'][-152:]

print(f"Starting CMS Evaluation on {len(test_threads)} test threads...")
# Look at the keys of the very first thread
print(test_threads[0].keys())

for i in tqdm(range(len(test_threads))):
    thread_obj = test_threads[i]

    try:
        # 1. Generate the prediction
        pred = run_cms_pipeline(thread_obj, model, tokenizer)

        # 2. Get the ground truth using the correct key we just found
        truth = thread_obj.get('edited_sum')

        if pred and truth:
            cms_predictions.append(pred)
            ground_truth_references.append(truth)

    except Exception as e:
        print(f"Error on thread {i}: {e}")
        continue

# Calculate the scores
results = rouge.compute(predictions=cms_predictions, references=ground_truth_references)
print("\nCORRECTED CMS ROUGE SCORES:", results)

Starting CMS Evaluation on 152 test threads...
dict_keys(['submission_id', 'subreddit', 'url', 'caption', 'raw_caption', 'score', 'author', 'created_utc', 'permalink', 'num_comments', 'comments_length', 'comments', 'month', 'clusters_auto', 'author_anon', 'opsum', 'csums', 'unedited_sum', 'edited_sum'])


  0%|          | 0/152 [00:00<?, ?it/s]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


  1%|          | 1/152 [00:14<35:33, 14.13s/it]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


...

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


 99%|█████████▉| 151/152 [28:30<00:11, 11.92s/it]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


100%|██████████| 152/152 [28:45<00:00, 11.35s/it]



CORRECTED CMS ROUGE SCORES: {'rouge1': np.float64(0.4342754109373221), 'rouge2': np.float64(0.1838320934098007), 'rougeL': np.float64(0.29438978310475317), 'rougeLsum': np.float64(0.3382616455645504)}


**Getting rogue scores for T5-Base(without image)**


In [52]:
def evaluate_saved_model(model_path, model_type="t5"):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    print(f"\n--- Loading {model_type.upper()} from {model_path} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # Automatically loads BART or T5 weights correctly
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

    cms_preds = []
    truths = []

    # We use a subset (e.g., 50 threads) if you are very short on time,
    # or the full 152 for paper-ready results.
    test_threads = data['threads'][-152:]

    for i in tqdm(range(len(test_threads))):
        thread_obj = test_threads[i]
        try:
            pred = run_cms_pipeline(thread_obj, model, tokenizer)
            truth = thread_obj.get('edited_sum')
            if pred and truth:
                cms_preds.append(pred)
                truths.append(truth)
        except Exception as e:
            continue

    results = rouge.compute(predictions=cms_preds, references=truths)
    # Multiply by 100 for readability (0.43 -> 43.0)
    final_scores = {k: round(v * 100, 2) for k, v in results.items()}
    return final_scores

In [53]:
import pandas as pd

# Define your model paths and labels
model_configs = [
    {"path": "/content/final_mredditsum_model_without_img_caption_20_epoch", "label": "T5-Base (No Img)"},
]

all_results = []

for config in model_configs:
    print(f"\n" + "="*50)
    print(f"EVALUATING: {config['label']}")
    print("="*50)

    # Run the evaluation
    scores = evaluate_saved_model(config['path'], model_type="auto")

    # Add label for the table
    scores['Model Name'] = config['label']
    all_results.append(scores)

# 2. Generate Final Comparison Table
df_results = pd.DataFrame(all_results)
# Reorder columns for readability
df_results = df_results[['Model Name', 'rouge1', 'rouge2', 'rougeL', 'rougeLsum']]

print("\n\n--- FINAL SYSTEM COMPARISON ---")
print(df_results)


EVALUATING: T5-Base (No Img)

--- Loading AUTO from /content/final_mredditsum_model_without_img_caption_20_epoch ---


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
  0%|          | 0/152 [00:00<?, ?it/s]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


  1%|          | 1/152 [00:07<18:36,  7.39s/it]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


...

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


 99%|█████████▉| 151/152 [21:59<00:10, 10.03s/it]

Stage 2: Summarizing OP (Caption + Text) and 8 comment clusters...
Stage 3: Synthesizing final summary...


100%|██████████| 152/152 [22:09<00:00,  8.74s/it]




--- FINAL SYSTEM COMPARISON ---
         Model Name  rouge1  rouge2  rougeL  rougeLsum
0  T5-Base (No Img)   41.19   14.72   24.66      30.03
